# 長い文書を分割して要約する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_summarization/long_document_summary.ipynb)

NLP クックブック 23-06 ／ 要約と平易化

モデルに一度に入らない (あるいは入れると後半を読み飛ばす) 長い文書を要約したい。分割して要約し、それをまとめ直す。

In [ ]:
import io, re, urllib.request, zipfile
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def chat(prompt, max_new_tokens=160):
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

text = aozora("https://www.aozora.gr.jp/cards/000035/files/1567_ruby_4948.zip")   # 走れメロス (約 1 万字)
sents = [s.strip() for s in re.split(r"(?<=[。！？」])(?![」。！？])|\n+", text) if s and s.strip()]

def make_chunks(sents, max_chars=2500):
    chunks, cur = [], ""
    for s in sents:
        if cur and len(cur) + len(s) > max_chars:
            chunks.append(cur); cur = ""
        cur += s
    return chunks + [cur]

chunks = make_chunks(sents)
print(len(text), "文字 →", len(chunks), "分割", [len(c) for c in chunks])

In [ ]:
# 1. 分割ごとに要約する (map)
partial = []
for i, chunk in enumerate(chunks, 1):
    s = chat(f"次は小説の一部です。この部分で起きた出来事を、登場人物の名前を入れて 2〜3 文で要約してください。要約だけを出力してください。\n\n{chunk}", max_new_tokens=140)
    partial.append(s)
    print(f"[{i}] {s}")

In [ ]:
# 2. 部分要約をつなげて、全体の要約を作る (reduce)
joined = "\n".join(f"{i}. {s}" for i, s in enumerate(partial, 1))
final = chat(f"次は、ある小説を前から順に分けて要約したものです。これらをまとめて、物語全体のあらすじを 4 文で書いてください。あらすじだけを出力してください。\n\n{joined}", max_new_tokens=220)
print(final)

- 「分割して要約 → 要約を要約」は map-reduce 型と呼ばれる。分割ごとの処理は独立なので、並列化もできる。
- 部分要約の段階で落とした情報は、最終要約には戻らない。固有名詞や数値を残すよう、部分要約のプロンプトで指示する。
- 前の部分の要約を次の部分の入力に加えていく方法 (逐次的な洗練) だと、話のつながりは保ちやすいが、並列化はできず、誤りが後ろに伝わる。
- 長い入力をそのまま読めるモデルでも、文書が長くなるほど中ほどの内容を見落としやすい。分割は今でも有効な方法。